# Method 1 — Part 2: pixel-wise linear heads (TESSERA, AlphaEarth, concatenation, noise control)

**Question:** with the simplest head (a logistic regression per pixel, no spatial context), does the concatenation TESSERA 2019 + AlphaEarth 2019 beat each embedding alone, and does it beat TESSERA + 64 channels of noise?

| Variant | Description |
|---|---|
| `t19` | TESSERA 2019 (128 ch.) |
| `ae19` | AlphaEarth 2019 (64 ch.) |
| `t19-ae19` | TESSERA 2019 + AlphaEarth 2019 concatenation (192 ch.) |
| `t19-noise` | TESSERA 2019 + 64 N(0, 1) noise channels (dimensionality control) |

For each of the seeds 0, 1, 2: 300,000 valid training pixels from the 1,455 training patches of fold I (the seed changes the pixel sample and the noise; all variants of a seed share them), class weights 1/√frequency, per-channel standardisation, logistic regression with up to 1,000 iterations, evaluation on the 496 test patches (shared valid mask). Summary: mean over the seeds with a patch-bootstrap interval; paired differences averaged over seeds with the same patch draws. An improvement is claimed only when the 95% interval excludes 0.

**Needs:** Parts 1–7 on the Drive. **Runtime:** CPU (High-RAM not needed); keep the tab open. **Time:** about 1–1.5 h (copy of about 10.6 GB, then about 20 min per seed).

**If the runtime disconnects:** run steps 1–3 again, then step 4: seeds already saved on the Drive are skipped.

## 1. Runtime, Drive and configuration

In [ ]:
import sys, shutil, platform
from pathlib import Path
import pandas as pd
print("Python", platform.python_version())
print(f"free space in /content: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/pj_vmi")
REPO_URL = "https://github.com/ColdCastlee/ProjectVMI.git"
BRANCH = "main"
CODE = Path("/content/code")
PROC_DRIVE = DRIVE / "processed"
PROC = Path("/content/processed")
RUNS = DRIVE / "runs"
RUNS_CSV = DRIVE / "results" / "runs.csv"
OUT = DRIVE / "results" / "m1"

## 2. Code and tests
**Check:** all tests pass (`15 passed` or more).

In [ ]:
if (CODE / ".git").exists():
    !git -C "{CODE}" fetch -q origin {BRANCH} && git -C "{CODE}" checkout -q {BRANCH} && git -C "{CODE}" pull -q --ff-only origin {BRANCH}
else:
    !git clone -q -b {BRANCH} {REPO_URL} "{CODE}"
%cd {CODE}
!git log -1 --format="commit %h  %ad  %s" --date=short
!pip install -q -r requirements.txt
!python -m pytest -q tests

## 3. Copy the data to the local disk
Reading the embeddings directly on the Drive is slow: copy them once per session.

**Check:** every line ends with `ok=True`.

In [ ]:
PROC.mkdir(parents=True, exist_ok=True)
for f in ["meta.csv", "splits.json", "labels.npy", "instances.npy", "valid_mask.npy",
          "tessera_2019_q.npy", "tessera_2019_scale.npy", "ae_2019.npy"]:
    if not (PROC / f).exists() or (PROC / f).stat().st_size != (PROC_DRIVE / f).stat().st_size:
        shutil.copy(PROC_DRIVE / f, PROC / f)
    print(f"{f:24s} {(PROC / f).stat().st_size / 1e9:6.2f} GB  ok={(PROC / f).stat().st_size == (PROC_DRIVE / f).stat().st_size}")

## 4. Train and evaluate, seeds 0, 1, 2
Each line `seed s variant: ... iterations (converged | max_iter reached)` is one model. The runs of a seed are saved on the Drive when the seed finishes.

**Check:** the three tables are printed (per seed, mean over seeds, paired differences), `results/runs.csv` holds the 12 runs `lin_<variant>_100_foldI_s<seed>`, and `results/m1/` holds `linear_heads_per_seed.csv`, `linear_heads_summary.csv`, `linear_heads_deltas.csv` and `linear_heads_per_class.png`.

**How to read it:**
- `all_converged` should be True; otherwise note which models stopped at 1,000 iterations;
- `t19-ae19` vs `t19`: is the interval of the mean difference above 0, and is the difference positive for every seed (`per_seed`)?
- `t19-ae19` vs `t19-noise`: the concatenation must beat the same number of random channels;
- `miou_std` (spread between seeds) compared with the differences between variants.

In [ ]:
!python scripts/m1_linear_heads.py --processed "{PROC}" --runs-dir "{RUNS}" --runs-csv "{RUNS_CSV}" --out "{OUT}" --seeds 0 1 2

In [ ]:
from IPython.display import Image, display
display(Image(str(OUT / "linear_heads_per_class.png")))